# clip-pipeline en Google Colab

Este notebook clona el repo, instala dependencias, pide la API key de Anthropic de forma segura y corre el pipeline completo (descarga -> transcripcion -> deteccion de momentos -> clips).

Todo el codigo real vive en `src/` y es agnostico del entorno: no tiene rutas de Colab hardcodeadas, asi que el mismo repo sirve despues para correr en un VPS.

## 1. Clonar el repo (o hacer `pull` si ya existe)

In [ ]:
import os

# Edita estos valores si tu repo o rama son distintos.
REPO_URL = "https://github.com/gvmaplicaciones/clippeando_zaloclips.git"
BRANCH = "main"
REPO_DIR = "/content/clip-pipeline"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("El repo ya existe, haciendo pull...")
    !git -C "$REPO_DIR" pull origin "$BRANCH"
else:
    print("Clonando el repo...")
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR

## 2. Instalar dependencias de `requirements.txt`

In [ ]:
!pip install -q -r requirements.txt

## 3. Verificar / instalar ffmpeg

La mayoria de las imagenes de Colab ya traen ffmpeg, pero lo verificamos e instalamos si hace falta.

In [ ]:
import shutil

if shutil.which("ffmpeg") is None:
    print("ffmpeg no encontrado, instalando...")
    !apt-get -qq update && apt-get -qq install -y ffmpeg
else:
    print("ffmpeg ya esta instalado:", shutil.which("ffmpeg"))

!ffmpeg -version | head -n 1

## 4. Instalar deno (runtime de JS que yt-dlp necesita para YouTube)

yt-dlp lo usa para resolver los desafios anti-bot de YouTube. Sin esto, la
extraccion puede fallar o caer en clientes (android/ios) que no respetan las
cookies de sesion.

In [ ]:
import os
import shutil

DENO_BIN = os.path.expanduser("~/.deno/bin")
os.environ["PATH"] = DENO_BIN + os.pathsep + os.environ.get("PATH", "")

if shutil.which("deno") is None:
    print("deno no encontrado, instalando...")
    !curl -fsSL https://deno.land/install.sh | sh -s -- -y
else:
    print("deno ya esta instalado:", shutil.which("deno"))

!deno --version

## 5. Configurar `ANTHROPIC_API_KEY` de forma segura

Esta celda pide la API key con un input **oculto** (no queda escrita en el notebook) y la guarda en un archivo `.env` local, que ya esta en `.gitignore` y nunca se sube al repo.

> **IMPORTANTE:** nunca pegues tu API key directamente como texto en una celda. Si por error la escribis en el codigo de una celda (en vez de tipearla en el prompt oculto de abajo), **borra esa celda y su output antes de hacer commit/push**, para no subir la key al repo.

In [ ]:
from getpass import getpass
from pathlib import Path

api_key = getpass("Pega tu ANTHROPIC_API_KEY (no se mostrara en pantalla): ")

env_path = Path(".env")
lines = []
if env_path.exists():
    lines = [l for l in env_path.read_text().splitlines() if not l.startswith("ANTHROPIC_API_KEY=")]
lines.append(f"ANTHROPIC_API_KEY={api_key}")
env_path.write_text("\n".join(lines) + "\n")

del api_key
print(".env actualizado. Recorda: .env esta en .gitignore, no se sube al repo.")

## 6. (Opcional) Autenticar la descarga si YouTube pide "Sign in to confirm you're not a bot"

YouTube suele bloquear descargas desde IPs de datacenter como las de Colab con
el error `Sign in to confirm you're not a bot`. Si te pasa, subi un
`cookies.txt` exportado de tu navegador (logueado en YouTube, con una
extension como "Get cookies.txt LOCALLY") usando el panel de archivos de la
izquierda, y corre esta celda. Si no te pasa el error, salteala.

In [ ]:
from pathlib import Path

cookies_path = "/content/cookies.txt"  # ajusta la ruta si subiste el archivo a otro lugar

if Path(cookies_path).exists():
    env_path = Path(".env")
    lines = []
    if env_path.exists():
        lines = [l for l in env_path.read_text().splitlines() if not l.startswith("YTDLP_COOKIES_FILE=")]
    lines.append(f"YTDLP_COOKIES_FILE={cookies_path}")
    env_path.write_text("\n".join(lines) + "\n")
    print(f".env actualizado con YTDLP_COOKIES_FILE={cookies_path}")
else:
    print(f"No se encontro {cookies_path} - subilo primero o saltea esta celda si no lo necesitas.")

## 7. Correr el pipeline

Ejemplo con una URL. Si preferis usar un archivo ya subido a `input/`, comenta la linea de `url` y descomenta la de `file`.

In [ ]:
from src.pipeline import run_pipeline

clips = run_pipeline(
    url="https://www.youtube.com/watch?v=EJEMPLO",
    # file="input/mi_video.mp4",
)

clips